# Batch Structured JSON Comparison

This notebook compares the `structured_json` outputs under `D:\\AQUMON\\test_batch_size\\batch4` and `D:\\AQUMON\\test_batch_size\\batch8`.

It reports:
- how many JSON files are shared or missing across the two runs
- how many files contain any field-level difference
- how many total field mismatch instances exist across all matched files
- which field paths differ most often
- detailed differences for selected files


In [1]:
from __future__ import annotations

import json
from collections import Counter
from pathlib import Path

import pandas as pd
from IPython.display import display

BATCH4_BASE = Path(r"D:\AQUMON\test_batch_size\batch4")
BATCH8_BASE = Path(r"D:\AQUMON\test_batch_size\batch8")

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_rows", 200)


In [2]:
MISSING = object()


def resolve_structured_json_root(base_dir: Path) -> Path:
    direct_root = base_dir / "structured_json"
    if direct_root.exists():
        return direct_root

    candidates = sorted(
        child / "structured_json"
        for child in base_dir.iterdir()
        if child.is_dir() and (child / "structured_json").exists()
    )

    if not candidates:
        raise FileNotFoundError(f"No structured_json directory found under {base_dir}")
    if len(candidates) > 1:
        raise ValueError(
            f"Multiple structured_json directories found under {base_dir}: {candidates}"
        )
    return candidates[0]


def collect_json_files(root: Path) -> dict[str, Path]:
    return {
        str(path.relative_to(root)).replace("\\", "/"): path
        for path in sorted(root.rglob("*.json"))
    }


def flatten_json(value, prefix: str = "") -> dict[str, object]:
    flat: dict[str, object] = {}

    if isinstance(value, dict):
        if prefix and not value:
            flat[prefix] = {}
        for key in sorted(value):
            next_prefix = f"{prefix}.{key}" if prefix else key
            flat.update(flatten_json(value[key], next_prefix))
        return flat

    if isinstance(value, list):
        length_key = f"{prefix}.__len__" if prefix else "__len__"
        flat[length_key] = len(value)
        for idx, item in enumerate(value):
            next_prefix = f"{prefix}[{idx}]" if prefix else f"[{idx}]"
            flat.update(flatten_json(item, next_prefix))
        return flat

    flat[prefix or "$"] = value
    return flat


def display_value(value: object) -> object:
    if value is MISSING:
        return "<MISSING>"
    if isinstance(value, (dict, list)):
        return json.dumps(value, ensure_ascii=False, sort_keys=True)
    return value


def compare_json_objects(left_obj: object, right_obj: object) -> list[dict[str, object]]:
    left_flat = flatten_json(left_obj)
    right_flat = flatten_json(right_obj)
    all_paths = sorted(set(left_flat) | set(right_flat))

    diffs: list[dict[str, object]] = []
    for field_path in all_paths:
        left_value = left_flat.get(field_path, MISSING)
        right_value = right_flat.get(field_path, MISSING)
        if left_value != right_value:
            diffs.append(
                {
                    "field_path": field_path,
                    "batch4_value": display_value(left_value),
                    "batch8_value": display_value(right_value),
                }
            )
    return diffs


def load_json(path: Path) -> object:
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


In [3]:
batch4_root = resolve_structured_json_root(BATCH4_BASE)
batch8_root = resolve_structured_json_root(BATCH8_BASE)

batch4_files = collect_json_files(batch4_root)
batch8_files = collect_json_files(batch8_root)

common_files = sorted(set(batch4_files) & set(batch8_files))
only_in_batch4 = sorted(set(batch4_files) - set(batch8_files))
only_in_batch8 = sorted(set(batch8_files) - set(batch4_files))

per_file_rows = []
field_counter = Counter()
field_example_rows = {}
diff_details_by_file = {}

for file_key in common_files:
    left_obj = load_json(batch4_files[file_key])
    right_obj = load_json(batch8_files[file_key])
    diffs = compare_json_objects(left_obj, right_obj)

    if diffs:
        diff_details_by_file[file_key] = pd.DataFrame(diffs)

    per_file_rows.append(
        {
            "file_key": file_key,
            "field_mismatch_count": len(diffs),
            "has_difference": bool(diffs),
        }
    )

    for diff in diffs:
        field_path = diff["field_path"]
        field_counter[field_path] += 1
        field_example_rows.setdefault(
            field_path,
            {
                "example_file": file_key,
                "example_batch4_value": diff["batch4_value"],
                "example_batch8_value": diff["batch8_value"],
            },
        )

file_diff_df = pd.DataFrame(per_file_rows).sort_values(
    ["field_mismatch_count", "file_key"],
    ascending=[False, True],
).reset_index(drop=True)

field_diff_df = pd.DataFrame(
    [
        {
            "field_path": field_path,
            "files_with_difference": count,
            **field_example_rows[field_path],
        }
        for field_path, count in field_counter.most_common()
    ]
)

summary_df = pd.DataFrame(
    [
        {
            "batch4_root": str(batch4_root),
            "batch8_root": str(batch8_root),
            "batch4_json_files": len(batch4_files),
            "batch8_json_files": len(batch8_files),
            "common_json_files": len(common_files),
            "only_in_batch4": len(only_in_batch4),
            "only_in_batch8": len(only_in_batch8),
            "files_with_any_difference": int(file_diff_df["has_difference"].sum()),
            "total_field_mismatch_instances": int(file_diff_df["field_mismatch_count"].sum()),
            "unique_field_paths_with_differences": len(field_counter),
        }
    ]
)

display(summary_df)


,batch4_root,batch8_root,batch4_json_files,batch8_json_files,common_json_files,only_in_batch4,only_in_batch8,files_with_any_difference,total_field_mismatch_instances,unique_field_paths_with_differences
0,D:\AQUMON\test_batch_size\batch4\4_extraction_v1_gemi_batch\structured_json,D:\AQUMON\test_batch_size\batch8\4_extraction_v1_gemi_batch\structured_json,54,54,54,0,0,54,310,100


In [4]:
display(file_diff_df)

if only_in_batch4:
    display(pd.DataFrame({"only_in_batch4": only_in_batch4}))

if only_in_batch8:
    display(pd.DataFrame({"only_in_batch8": only_in_batch8}))


,file_key,field_mismatch_count,has_difference
0,TDG/0000919574-20-002399/0000919574-20-002399.json,32,True
1,SPGI/0000064040-23-000067/0000064040-23-000067.json,20,True
2,YOU/0001209191-22-028778/0001209191-22-028778.json,16,True
3,INCY/0001209191-20-025112/0001209191-20-025112.json,15,True
4,KALU/0001209191-22-045612/0001209191-22-045612.json,13,True
5,PSX/0001534701-20-000072/0001534701-20-000072.json,13,True
6,IRM/0001020569-22-000056/0001020569-22-000056.json,12,True
7,KR/0001209191-20-019112/0001209191-20-019112.json,12,True
8,LOVE/0001213900-24-034233/0001213900-24-034233.json,12,True
9,COF/0000927628-21-000067/0000927628-21-000067.json,10,True


In [5]:
display(field_diff_df)


,field_path,files_with_difference,example_file,example_batch4_value,example_batch8_value
0,filing_date,36,ADSK/0001209191-20-021012/0001209191-20-021012.json,2020-03-24,None
1,table_i_non_derivative[0].transaction_form_type_indicator_v,20,ADSK/0001209191-20-021012/0001209191-20-021012.json,False,None
2,table_i_non_derivative[1].transaction_form_type_indicator_v,12,ADSK/0001209191-20-021012/0001209191-20-021012.json,False,None
3,remarks,11,AGO/0001209191-20-021889/0001209191-20-021889.json,"/s/ Ling Chow, Attorney-in-fact",None
4,table_i_non_derivative[2].transaction_form_type_indicator_v,8,AGO/0001209191-20-021889/0001209191-20-021889.json,False,None
5,period_of_report,8,BKR/0001701605-18-000067/0001701605-18-000067.json,None,2018-05-11
6,filing_characteristics.rule_10b5_1,8,CFG/0000759944-20-000054/0000759944-20-000054.json,None,False
7,date_of_earliest_transaction,8,INCY/0001209191-20-025112/0001209191-20-025112.json,04/16/2020,2020-04-16
8,table_i_non_derivative[0].transaction_date,8,INCY/0001209191-20-025112/0001209191-20-025112.json,04/16/2020,2020-04-16
9,table_i_non_derivative[0].transaction_code_footnote_refs.__len__,7,ARW/0001127602-20-010778/0001127602-20-010778.json,1,0


In [6]:
def show_file_differences(file_key: str, max_rows: int = 200) -> pd.DataFrame:
    if file_key not in diff_details_by_file:
        raise KeyError(f"No differences recorded for {file_key}")
    return diff_details_by_file[file_key].head(max_rows)


differing_files = file_diff_df[file_diff_df["field_mismatch_count"] > 0].copy()
display(differing_files)

if not differing_files.empty:
    example_file_key = differing_files.iloc[0]["file_key"]
    print(f"Example file with differences: {example_file_key}")
    display(show_file_differences(example_file_key))


,file_key,field_mismatch_count,has_difference
0,TDG/0000919574-20-002399/0000919574-20-002399.json,32,True
1,SPGI/0000064040-23-000067/0000064040-23-000067.json,20,True
2,YOU/0001209191-22-028778/0001209191-22-028778.json,16,True
3,INCY/0001209191-20-025112/0001209191-20-025112.json,15,True
4,KALU/0001209191-22-045612/0001209191-22-045612.json,13,True
5,PSX/0001534701-20-000072/0001534701-20-000072.json,13,True
6,IRM/0001020569-22-000056/0001020569-22-000056.json,12,True
7,KR/0001209191-20-019112/0001209191-20-019112.json,12,True
8,LOVE/0001213900-24-034233/0001213900-24-034233.json,12,True
9,COF/0000927628-21-000067/0000927628-21-000067.json,10,True


Example file with differences: TDG/0000919574-20-002399/0000919574-20-002399.json


,field_path,batch4_value,batch8_value
0,date_of_earliest_transaction,03/10/2020,2020-03-10
1,filing_date,03/12/2020,2020-03-12
2,table_i_non_derivative[0].transaction_date,03/11/2020,2020-03-11
3,table_i_non_derivative[10].transaction_date,03/11/2020,2020-03-11
4,table_i_non_derivative[11].transaction_date,03/11/2020,2020-03-11
5,table_i_non_derivative[12].transaction_date,03/11/2020,2020-03-11
6,table_i_non_derivative[13].transaction_date,03/11/2020,2020-03-11
7,table_i_non_derivative[14].transaction_date,03/11/2020,2020-03-11
8,table_i_non_derivative[15].transaction_date,03/11/2020,2020-03-11
9,table_i_non_derivative[16].transaction_date,03/11/2020,2020-03-11
